# Drone Thermal Object Detection — HIT-UAV (Colab)

This notebook prepares the standard HIT-UAV annotations, trains one YOLOv8 model from generic pretrained weights, evaluates on the official test split, and downloads a separate checkpoint. It does not overwrite the existing FLIR checkpoint.

The trained model detects labeled objects; border-specific anomaly decisions require additional data or temporal logic.

In [ ]:
!pip -q install kaggle ultralytics pyyaml

from pathlib import Path
import json, os, shutil, yaml

OUT_DIR = Path('/content/processed_hit_uav')
OUT_DIR.parent.mkdir(parents=True, exist_ok=True)

## Download HIT-UAV from Kaggle
Enter your Kaggle API token when prompted. The token input is masked and is not written to a file. Generate it from Kaggle Account Settings → API → Generate New Token. This downloads the HIT-UAV Kaggle dataset directly; Kaggle errors stop the notebook instead of being treated as a successful download.

In [ ]:
from getpass import getpass
import subprocess

DOWNLOAD_DIR = Path('/content/hit_uav_download')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
kaggle_token = getpass('Kaggle API token: ')
if not kaggle_token.strip():
    raise ValueError('A Kaggle API token is required to download this dataset.')
os.environ['KAGGLE_API_TOKEN'] = kaggle_token.strip()
dataset_slug = 'pandrii000/hituav-a-highaltitude-infrared-thermal-dataset'
try:
    subprocess.run(['kaggle', 'datasets', 'download', '-d', dataset_slug,
                    '-p', str(DOWNLOAD_DIR), '--unzip'], check=True)
finally:
    os.environ.pop('KAGGLE_API_TOKEN', None)

print('Kaggle download completed under', DOWNLOAD_DIR)

## Prepare Kaggle's YOLO folders
Discover the dataset YAML, preserve its train/validation/test splits, and remap class IDs by name. `DontCare` is excluded; the output order is person, car, bicycle, other_vehicle.

In [ ]:
SPLITS = ('train', 'val', 'test')
CLASS_NAMES = ['person', 'car', 'bicycle', 'other_vehicle']
TARGET_ID = {'person': 0, 'car': 1, 'bicycle': 2, 'other vehicle': 3,
             'othervehicle': 3, 'othervechicle': 3, 'other_vehicle': 3}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def normalized_name(name):
    return ' '.join(str(name).casefold().replace('_', ' ').split())

def resolve_split_dir(value, yaml_path, config, split):
    value_path = Path(str(value))
    config_root = Path(str(config.get('path', '.')))
    if not config_root.is_absolute(): config_root = (yaml_path.parent / config_root).resolve()
    candidates = []
    if value_path.is_absolute(): candidates.append(value_path)
    else:
        candidates.extend([config_root / value_path, yaml_path.parent / value_path, DOWNLOAD_DIR / value_path])
    for candidate in candidates:
        if candidate.is_dir() and any(p.suffix.lower() in IMAGE_SUFFIXES for p in candidate.rglob('*')):
            return candidate.resolve()
    matches = [p for p in DOWNLOAD_DIR.rglob(split) if p.is_dir() and p.parent.name.lower() == 'images']
    if len(matches) == 1: return matches[0].resolve()
    raise FileNotFoundError(f'Cannot resolve {split} image folder from {value!r}; checked {candidates}')

def find_dataset_yaml():
    candidates = []
    for path in list(DOWNLOAD_DIR.rglob('*.yaml')) + list(DOWNLOAD_DIR.rglob('*.yml')):
        try: config = yaml.safe_load(path.read_text()) or {}
        except Exception: continue
        if isinstance(config, dict) and all(key in config for key in ('train', 'val', 'names')):
            candidates.append((path, config))
    usable = []
    for path, config in candidates:
        try:
            split_dirs = {split: resolve_split_dir(config[split], path, config, split) for split in SPLITS}
            usable.append((path, config, split_dirs))
        except FileNotFoundError:
            continue
    if len(usable) != 1:
        found = [str(path) for path, _ in candidates]
        downloaded_paths = [str(p.relative_to(DOWNLOAD_DIR)) for p in list(DOWNLOAD_DIR.rglob('*'))[:40]]
        raise RuntimeError(f'Expected one YAML with usable train/val/test image folders. Candidate YAMLs: {found}; files under download: {downloaded_paths}')
    return usable[0]

source_yaml, source_config, source_images = find_dataset_yaml()
source_names = source_config['names']
if isinstance(source_names, dict): source_names = [source_names[key] for key in sorted(source_names, key=lambda item: int(item))]
source_names = [str(name) for name in source_names]
source_to_target = {}
for source_id, name in enumerate(source_names):
    canonical = normalized_name(name)
    if canonical in TARGET_ID: source_to_target[source_id] = TARGET_ID[canonical]
missing_classes = [name for name in CLASS_NAMES if TARGET_ID[name] not in source_to_target.values()]
if missing_classes: raise ValueError(f'Kaggle YAML is missing target classes {missing_classes}; classes found: {source_names}')
print('Using Kaggle YAML:', source_yaml)
print('Source classes:', dict(enumerate(source_names)))
print('Source image folders:', source_images)

if OUT_DIR.exists(): shutil.rmtree(OUT_DIR)
split_counts = {}
for split in SPLITS:
    source_image_dir = source_images[split]
    source_label_dir = source_image_dir.parent.parent / 'labels' / split
    if not source_label_dir.is_dir():
        candidates = [p for p in DOWNLOAD_DIR.rglob(split) if p.is_dir() and p.parent.name.lower() == 'labels']
        if len(candidates) != 1: raise FileNotFoundError(f'Cannot locate labels/{split}; candidates: {candidates}')
        source_label_dir = candidates[0]
    target_image_dir, target_label_dir = OUT_DIR / split / 'images', OUT_DIR / split / 'labels'
    target_image_dir.mkdir(parents=True, exist_ok=True); target_label_dir.mkdir(parents=True, exist_ok=True)
    image_files = sorted(p for p in source_image_dir.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
    target_boxes = 0
    for image_path in image_files:
        label_path = source_label_dir / f'{image_path.stem}.txt'
        dst_image = target_image_dir / image_path.name
        shutil.copy2(image_path, dst_image)
        kept = []
        if label_path.is_file():
            for line_number, line in enumerate(label_path.read_text().splitlines(), start=1):
                if not line.strip(): continue
                fields = line.split()
                if len(fields) != 5: raise ValueError(f'Expected YOLO detection row in {label_path}:{line_number}; got {line!r}')
                source_id = int(float(fields[0]))
                if source_id in source_to_target:
                    fields[0] = str(source_to_target[source_id])
                    kept.append(' '.join(fields))
        target_boxes += len(kept)
        (target_label_dir / f'{image_path.stem}.txt').write_text('\n'.join(kept) + ('\n' if kept else ''))
    split_counts[split] = (len(image_files), target_boxes)
    print(f'{split}: {len(image_files)} images, {target_boxes} retained boxes')

data_yaml = OUT_DIR / 'hit_uav.yaml'
data_yaml.write_text('path: ' + json.dumps(str(OUT_DIR)) + '\n' +
                     'train: train/images\nval: val/images\ntest: test/images\n' +
                     f'nc: {len(CLASS_NAMES)}\nnames: {json.dumps(CLASS_NAMES)}\n')
print('Prepared dataset config:', data_yaml.read_text())

## Train and evaluate
This is one HIT-UAV run from generic YOLOv8 pretrained weights. Validation uses the provided validation split; final metrics use the held-out test split.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(data=str(data_yaml), epochs=50, imgsz=640, workers=2, batch=16,
            project='/content/aerial-thermal', name='hit_uav_yolov8n',
            seed=42, patience=15)

best_path = Path('/content/aerial-thermal/hit_uav_yolov8n/weights/best.pt')
model = YOLO(str(best_path))
test_metrics = model.val(data=str(data_yaml), split='test', plots=True)
print('Test mAP50:', float(test_metrics.box.map50))
print('Test mAP50-95:', float(test_metrics.box.map))
per_class = {}
for i, class_id in enumerate(test_metrics.box.ap_class_index):
    per_class[CLASS_NAMES[int(class_id)]] = {
        'precision': float(test_metrics.box.p[i]),
        'recall': float(test_metrics.box.r[i]),
        'AP50': float(test_metrics.box.all_ap[i, 0]),
    }
print('Per-class metrics:', per_class)

## Optional FLIR checkpoint comparison
Upload your existing FLIR `best.pt` if you want to inspect how it predicts on the same test images. Its class set has three classes while HIT-UAV has four, so treat this as a qualitative transfer check rather than a directly comparable four-class mAP score. The FLIR file is only read; the HIT-UAV checkpoint is saved under a distinct name.

In [ ]:
from google.colab import files
from IPython.display import display
from PIL import Image

flir_upload = files.upload()
if flir_upload:
    flir_path = Path('/content/flir_best.pt')
    flir_path.write_bytes(next(iter(flir_upload.values())))
    flir_model = YOLO(str(flir_path))
    sample_images = sorted((OUT_DIR / 'test' / 'images').glob('*.jpg'))[:8]
    for image_path in sample_images:
        print(image_path.name, 'FLIR classes:', flir_model.names)
        display(Image.fromarray(flir_model.predict(str(image_path), verbose=False)[0].plot()[..., ::-1]))
else:
    print('Skipped FLIR comparison; the existing local checkpoint remains unchanged.')

## Download the HIT-UAV weights
Save as `ai-model/weights/hit_uav_best.pt` locally. Keep your FLIR checkpoint separately as `best.pt` for reference. Cite the HIT-UAV paper and dataset when reporting results.

In [ ]:
files.download(str(best_path))